
# Streamlit 
## Bonus : Deep Learning 
### Objectif
L'objectif de ce notebook Bonus est de réaliser un projet complet de Deep Learning et de le présenter intéractivement grâce à Streamlit. Cela permettra d'appliquer les "bonnes pratiques" présentées dans le troisième notebook et de les adapter au Deep Learning.

Dans ce projet, nous allons construire un modèle Word2Vec, c'est-à-dire un modèle permettant de faire du prolongement lexical (word embeddings). Nous allons entrainer le modèle, le sauvegarder et présenter ses résultats en déployant une application Streamlit hébergée.

Comme précedemment, les différentes étapes du projet sont à réaliser en local sur un environnement de travail personnel (Jupyter Notebook, Google Colab ou éditeur de code) et non pas sur la plateforme. Le dataframe est téléchargeable au lien suivant.

### Données
Nous avons à notre disposition des données constituées de 25000 critiques de films. L'objectif est d'entrainer les matrices de word embeddings sur ces données.

(a) Charger le fichier de données sous le nom df et l'explorer.
Puisque l'approche Word2Vec ne nécessite que du texte, nous n'avons pas besoin de la colonne "sentiment" du dataframe.

(b) Supprimer la colonne "sentiment" de df.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("MovieReview.csv")

In [3]:
display(df.head())
print(df.shape)

df = df.drop('sentiment', axis=1)

,sentiment,review
0,Positive,With all this stuff going down at the moment w...
1,Positive,'The Classic War of the Worlds' by Timothy Hin...
2,Negative,The film starts with a manager (Nicholas Bell)...
3,Negative,It must be assumed that those who praised this...
4,Positive,Superbly trashy and wondrously unpretentious 8...


(25000, 2)


(c) Ajouter le code suivant pour nettoyer les données et supprimer les stopwords.

In [4]:
import re
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Téléchargement non bloquant, sans interface graphique
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # requis depuis NLTK 3.8.2+

stop_words = stopwords.words('english')


In [5]:

# Converts the unicode file to ascii
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    # creating a space between a word and the punctuation following it
    # eg: "he is a boy." => "he is a boy ."
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!]+", " ", w)
    w = re.sub(r'\b\w{0,2}\b', '', w)

    # remove stopword
    mots = word_tokenize(w.strip())
    mots = [mot for mot in mots if mot not in stop_words]
    return ' '.join(mots).strip()

df.review = df.review.apply(lambda x :preprocess_sentence(x))
df.head()

,review
0,stuff going moment started listening music wat...
1,classic war worlds timothy hines entertaining ...
2,film starts manager nicholas bell giving welco...
3,must assumed praised film greatest filmed oper...
4,superbly trashy wondrously unpretentious explo...


### Tokens
La classe Tokenizer de tensorflow.keras.preprocessing.text permet de vectoriser un corpus de texte. En effet, il transforme chaque texte en une séquence d'entiers, chaque entier étant l'index d'un token dans un dictionnaire. L'argument num_words limite la taille du dictionnaire.

La méthode fit_on_texts permet de mettre à jour le dictionnaire à partir d'une liste de textes.

(d) Définir un objet tokenizer à l'aide du constructeur Tokenizer de tensorflow.keras.preprocessing.text en précisant une limite de mots du dictionnaire de 10000.
(e) Mettre à jour le dictionnaire du tokenizer à l'aide de la méthode fit_on_texts.

In [6]:
import tensorflow as tf
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df.review)

(e) Stocker le dictionnaire de correspondance entre mots et index dans la variable word2idx, et le dictionnaire de correspondance entre index et mot dans la variable idx2word, à l'aide de l'attribut word_index du tokenizer.
(f) Stocker la taille du dictionnaire dans la variable vocab_size à l'aide de l'attribut num_words du tokenizer.

In [7]:
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = tokenizer.num_words

In [8]:
# ajouté par Emilie avec l'aide de IA pour récupérer vocab_size, word2idx et idx2word  dans le fichier .py de streamlit
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("num_words =", tokenizer.num_words, "| mots vus =", len(tokenizer.word_index))

num_words = 10000 | mots vus = 72540


### Modélisation
Nous allons implémenter un modèle Word2Vec particulier : le modèle Continuous Bag Of Words (CBOW). Le modèle CBOW cherche à prédire un mot grâce à son contexte, c'est-à-dire grâce aux mots proches de lui dans le texte. Les inputs du modèle sont les mots du contexte et l'output du modèle est une probabilité de prédiction du mot cible.

Plus précisemment, le modèle CBOW est un réseau de neurones à 3 couches : une couche d'input, une couche cachée et une couche ouput. La couche cachée est constituée d'une couche Embedding qui transforme chaque mot input en un vecteur d'embedding, de tel sorte que la matrice d'embedding est apprise au fur et à mesure de l'entrainement. Il y a aussi une couche de Pooling (GlobalAveragePooling1D) qui somme les différents embeddings pour obtenir un résultat de bonne dimension. Enfin la prédiction du mot cible est faite grâce à une couche Dense.

(g) Ajouter le code suivant pour créer l'ensemble de données (X, Y).

In [11]:
import numpy as np


def sentenceToData(tokens, WINDOW_SIZE):
    window = np.concatenate((np.arange(-WINDOW_SIZE,0),np.arange(1,WINDOW_SIZE+1)))
    X,Y=([],[])
    for word_index, word in enumerate(tokens) :
        if ((word_index - WINDOW_SIZE >= 0) and (word_index + WINDOW_SIZE <= len(tokens) - 1)) :
            X.append(word)
            Y.append([tokens[word_index-i] for i in window])
    return X, Y


WINDOW_SIZE = 5

X, Y = ([], [])
for review in df.review:
    for sentence in review.split("."):
        word_list = tokenizer.texts_to_sequences([sentence])[0]
        if len(word_list) >= WINDOW_SIZE:
            Y1, X1 = sentenceToData(word_list, WINDOW_SIZE//2)
            X.extend(X1)
            Y.extend(Y1)
    
X = np.array(X).astype(int)
y = np.array(Y).astype(int).reshape([-1,1])

(h) Créer l'architecture du modèle. La couche Embedding prendra une entrée de taille 10000 et une sortie de taille 300. La couche Dense sera constituée de 10000 neurones et d'une fonction d'activation SoftMax.

In [13]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

embedding_dim = 300
model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(GlobalAveragePooling1D())
model.add(Dense(vocab_size, activation='softmax'))

(i) Compiler le modèle.
(j) Entrainer le modèle sur 50 epochs.
Remarque : Le modèle est très lourd et l'entrainement peut prendre plusieurs heures.

In [14]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, batch_size = 128, epochs=50)

Epoch 1/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 5234s 430ms/step - accuracy: 0.0313 - loss: 7.6509
Epoch 2/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 541s 45ms/step - accuracy: 0.0581 - loss: 6.9537
Epoch 3/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 563s 46ms/step - accuracy: 0.0753 - loss: 6.5211
Epoch 4/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 574s 47ms/step - accuracy: 0.0886 - loss: 6.1901
Epoch 5/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 564s 46ms/step - accuracy: 0.1002 - loss: 5.9196
Epoch 6/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 620s 51ms/step - accuracy: 0.1108 - loss: 5.6969
Epoch 7/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 592s 49ms/step - accuracy: 0.1208 - loss: 5.5143
Epoch 8/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 594s 49ms/step - accuracy: 0.1301 - loss: 5.3654
Epoch 9/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 606s 50ms/step - accuracy: 0.1387 - loss: 5.2430
Epoch 10/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 583s 48ms/step - accuracy: 0.1464 - loss: 5.1424
Epoch 11/50
12163/12163 ━━━━━━━━━━━━━━━━━━━━ 586s 48ms/step - accur

### Enregistrement du modèle
Comme expliqué dans le troisième notebook, il est important de sauvegarder le modèle pour éviter qu'il ne soit ré-entrainé à chaque déploiement du Streamlit. C'est d'autant plus important en Deep Learning, où les modèles sont lourds et où leur entrainement prend du temps.

Avec Keras, il est possible d'enregistrer un modèle entrainé en entier (architecture, poids/filtres appris lors de l'entrainement et informations de compilation). Cela se fait grâce au format H5.

(g) Sauvegarder le modèle model au format H5 en utilisant la méthode save de Keras.

In [15]:
model.save("word2vec.h5") 

# SUITE A FAIRE DANS UN FICHIER .PY

### Création du fichier Python pour Streamlit
Maintenant que le modèle de Deep Learning a été entrainé et sauvegardé, nous pouvons passer au Streamlit. Comme d'habitude, nous utilisons un éditeur de code Python (par exemple VSCode ou Spyder) pour obtenir un fichier .py.

(h) Créer un fichier .py qui contiendra le script Python dédié à l'application Streamlit. L'enregistrer dans le même dossier que les autres fichiers du projet.
(i) Donner un titre au Streamlit.
(j) Charger les poids du modèle enregistré en utilisant la méthode load_weights de Keras.

In [ ]:
import streamlit as st
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

st.title("Modèle Word2Vec")

embedding_dim = 300
model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(GlobalAveragePooling1D())
model.add(Dense(vocab_size, activation='softmax'))

model.load_weights("word2vec.h5")

La similitude est une métrique mesurant la distance entre deux mots. Cette distance représente la façon dont les mots sont liés entre eux.

(k) Ajouter le code suivant pour extraire la matrice d'embeddings et définir les fonctions de similitude.

In [ ]:
vectors = model.layers[0].trainable_weights[0].numpy()
import numpy as np
from sklearn.preprocessing import Normalizer

def dot_product(vec1, vec2):
    return np.sum((vec1*vec2))

def cosine_similarity(vec1, vec2):
    return dot_product(vec1, vec2)/np.sqrt(dot_product(vec1, vec1)*dot_product(vec2, vec2))

def find_closest(word_index, vectors, number_closest):
    list1=[]
    query_vector = vectors[word_index]
    for index, vector in enumerate(vectors):
        if not np.array_equal(vector, query_vector):
            dist = cosine_similarity(vector, query_vector)
            list1.append([dist,index])
    return np.asarray(sorted(list1,reverse=True)[:number_closest])

def compare(index_word1, index_word2, index_word3, vectors, number_closest):
    list1=[]
    query_vector = vectors[index_word1] - vectors[index_word2] + vectors[index_word3]
    normalizer = Normalizer()
    query_vector =  normalizer.fit_transform([query_vector], 'l2')
    query_vector= query_vector[0]
    for index, vector in enumerate(vectors):
        if not np.array_equal(vector, query_vector):
            dist = cosine_similarity(vector, query_vector)
            list1.append([dist,index])
    return np.asarray(sorted(list1,reverse=True)[:number_closest])

def print_closest(word, number=10):
    index_closest_words = find_closest(word2idx[word], vectors, number)
    for index_word in index_closest_words :
        print(idx2word[index_word[1]]," -- ",index_word[0])

(l) Créer des widgets Streamlit permettant à l'utilisateur d'afficher les 10 mots les plus proches d'un mot choisi, grâce à la fonction print_closest définie précédemment.
Remarque : Une idée pour rendre le Streamlit encore plus intéractif pourrait être de laisser à l'utilisateur le choix du nombre de mots proches.

In [ ]:
#Exemple d'utilisation de la fonction print_closest
print_closest('zombie')

(m) Créer des widgets Streamlit pour jouer sur les propriétés sémantiques et arithmétiques d'un mot, préservées par le modèle Word2Vec. La fonction compare définie précedemment pourra être utilisée.
(n) Personnaliser le Streamlit.

### Déploiement de l'application Streamlit via GitHub
Nous voulons héberger notre application Streamlit. Pour cela, nous avons plusieurs possibilités :

via GitHub
via Colab
Nous essayons avec GitHub.

(o) Suivre les étapes du notebook précédent pour héberger l'application Streamlit à partir de GitHub. Créer notamment un repository GitHub et y déposer les fichiers nécessaires à l'application Streamlit (le modèle entrainé sauvegardé au format H5 et le fichier .py avec le script Streamlit).
A noter que si les poids d'un modèle de Deep Learning sont trop volumineux, le fichier H5 contenant le modèle ne peut pas être déposé sur le repo GitHub. Dans ce cas, nous pouvons utiliser Git Large File Storage (Git LFS), une fonctionnalité de Git permettant de stocker des fichiers lourds dans un dépôt distant. Cette fonctionnalité permet ainsi d'utiliser le flux de travail Git quelques soient les fichiers utilisés (données volumineuses, vidéos, songs, poids d'un modèle).

Pour utiliser Git LFS, il faut d'abord télécharger l'extension de commandes Git.

Puis :

git lfs install pour installer Git LFS
git clone pour cloner le repository GitHub
git lfs track "*.h5" pour utiliser Git LFS sur des fichiers en format H5
git add .gitattributes
Enfin, il faut utiliser les commandes Git classiques :

git add fichier.h5
git commit -m fichier.h5
git push

Maintenant vous maitrisez Streamlit et ses bonnes pratiques pour tout type de projet et pour tout environnement de travail !

### Déploiement de l'application Streamlit via Colab
Dans le cas où nous travaillons sur Google Colab, plutôt que de télécharger nos fichiers et de créer un repo GitHub, nous pouvons directement déployer une application Streamlit via Colab grâce à ngrok. ngrok est un proxy permettant de passer d'un URL public à un réseau privé (dans notre cas notre ordinateur en local).

Pour utiliser Streamlit via Google Colab avec ngrok, il faut suivre ces différentes étapes :

Installer Streamlit et ngrok sur Google Colab avec
!pip install -q streamlit
!pip install pyngrok
Créer un compte ngrok sur le site officiel
Copier le token disponible dans l'onglet "Your Authtoken"
Executer le code suivant sur Google Colab :
!./ngrok authtokens token #où token est le token copié
from pyngrok import ngrok 
public_url = ngrok.connect(port='8501')
public_url
Executer le code suivant dans une nouvelle cellule Google Colab pour créer un fichier .py (appelé streamlit_app.py ici). Puis écrire le script Python associé à l'application Streamlit dans la cellule.
%%writefile streamlit_app.py 
import streamlit as st 
#Insérer code Python contenant les commandes Streamlit
Une fois que le code est terminé, executer le code suivant dans une nouvelle cellule Google Colab pour déployer l'application Streamlit.
!streamlit run /content/streamlit_app.py & npx localtunnel — port 8501